[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/milestones/Milestone3_Model_Adaptation.ipynb)

> **Run this in Google Colab.** Click the badge above (or open the notebook from Canvas), then run the **Setup** cell first — it installs the dependencies. No local install needed.

# Assignment 3 · Milestone 3 — Model Adaptation Experiment
**ISBA 2411 · Due Sunday, July 26, 2026 (11:59 PM PT) · Week 6**

Experiment with adapting a model to your team's task and report what you learn.

## Deliverables
- **Completed notebook** — this notebook run on *your team's* data (demo data replaced at the swap-in cell).
- **500-word analysis** — which adaptation strategy best fits your project, and why.

## What to submit on Canvas
Submit **both** — the live work *and* an archival copy for the record:
1. **GitHub notebook link** — the URL to your team's completed notebook in your project repo (paste it in the *Website URL* box).
2. **PDF copy** — export the completed notebook to PDF (in Colab: *File → Print → Save as PDF*) and upload it (*File Upload*, PDF only).

> Canvas may only accept one submission method per attempt. Either way, **paste your GitHub notebook URL in the cell field below before exporting** so the link is also captured inside the PDF.

**Team GitHub notebook URL:** _(paste here before exporting to PDF)_

**Reading connections**

| Strategy | Where to read |
|---|---|
| Prompt engineering (zero-/few-shot) | HOLLM Ch. 6 |
| Fine-tuning representation models | HOLLM Ch. 11; Tunstall Ch. 2 |
| Fine-tuning generative models / PEFT (LoRA) | HOLLM Ch. 12 |
| Working with few labels | Tunstall Ch. 9 |

> Zero-shot, few-shot, and LoRA are the **default set**. Compare them on performance, cost, and effort.

> **How this notebook works.** It runs **end-to-end on a small built-in demo dataset** so you can see every step working before adapting it. Find the cell marked **`# ====== SWAP IN YOUR TEAM'S DATA HERE ======`** and replace the demo data with your own — everything downstream is written to work off the same variables.

> **Compute note.** The demo uses small models (`flan-t5-small`, `distilbert-base-uncased`, `all-MiniLM-L6-v2`) that run on Colab CPU; a GPU runtime is faster for the fine-tuning step. A fixed seed is set in setup.

### How this is graded
Scored on the shared milestone rubric ([`docs/ISBA2411_Assignment_Rubric.pdf`](../docs/ISBA2411_Assignment_Rubric.pdf)):

| Rubric criterion | In this milestone |
|---|---|
| **Execution** | Three working adaptation strategies on the same task + metric |
| **Analysis & Insight** | A reasoned cost/quality/effort comparison and a recommendation |
| **Communication** | Clear analysis tied to the prompting vs. fine-tuning readings |


In [ ]:
# ===== Setup (Colab) =====
!pip install -q transformers datasets peft accelerate scikit-learn pandas

import time, numpy as np, pandas as pd, torch, random
from sklearn.metrics import accuracy_score, f1_score
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

## 0. Task + data
Task: **binary sentiment** (positive / negative) on short product reviews. The metric is **accuracy** (plus macro-F1).

**To use your own data:** replace the demo block so you have `train` and `test` lists of `(text, label)` pairs with integer labels (`0`/`1`) and a `LABELS` name map. Keep the variable names and the three strategies below run unchanged.

In [ ]:
# ====== SWAP IN YOUR TEAM'S DATA HERE ======
LABELS = {0: 'negative', 1: 'positive'}
train = [
  ('Absolutely loved it, works perfectly and great value', 1),
  ('Fantastic quality, exceeded my expectations', 1),
  ('Best purchase I have made all year', 1),
  ('Super easy to use and looks beautiful', 1),
  ('Five stars, would buy again without hesitation', 1),
  ('Works exactly as described, very happy', 1),
  ('Comfortable, durable, and reasonably priced', 1),
  ('The setup was effortless and it runs great', 1),
  ('Terrible, broke after two days of use', 0),
  ('Complete waste of money, do not buy', 0),
  ('Cheap materials and stopped working quickly', 0),
  ('Very disappointed, nothing like the photos', 0),
  ('Awful experience, it arrived defective', 0),
  ('Stopped charging within a week, useless', 0),
  ('Poor build quality and overpriced', 0),
  ('Hated it, returned it the same day', 0),
]
test = [
  ('Really impressed with how well this works', 1),
  ('Great product, highly recommend to everyone', 1),
  ('Solid value and pleasant to use daily', 1),
  ('Exceeded expectations, very satisfied', 1),
  ('Pleasantly surprised by the quality', 1),
  ('Broke almost immediately, total junk', 0),
  ('Do not waste your money on this', 0),
  ('Flimsy and stopped working fast', 0),
  ('Worst purchase, deeply regret it', 0),
  ('Arrived damaged and never worked', 0),
]
test_texts  = [t for t, _ in test]
test_labels = [y for _, y in test]
print(f'train={len(train)}  test={len(test)}')

## 1. Strategy A — Zero-shot prompting
Prompt an instruction-tuned model (`flan-t5-small`) with **no examples**.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

flan_name = 'google/flan-t5-small'
flan_tok = AutoTokenizer.from_pretrained(flan_name)
flan = AutoModelForSeq2SeqLM.from_pretrained(flan_name).to(DEVICE)

def flan_generate(prompt, max_new_tokens=5):
    ids = flan_tok(prompt, return_tensors='pt').input_ids.to(DEVICE)
    out = flan.generate(ids, max_new_tokens=max_new_tokens)
    return flan_tok.decode(out[0], skip_special_tokens=True)

def parse_sentiment(out):
    return 1 if 'positive' in out.lower() else 0

def zero_shot(text):
    prompt = (f'Is the sentiment of this product review positive or negative?\n'
              f'Review: "{text}"\nAnswer:')
    return parse_sentiment(flan_generate(prompt))

t0 = time.time()
pred_zs = [zero_shot(t) for t in test_texts]
zs = dict(strategy='zero-shot',
          accuracy=round(accuracy_score(test_labels, pred_zs), 3),
          macro_f1=round(f1_score(test_labels, pred_zs, average='macro'), 3),
          sec=round(time.time()-t0, 1), trained_params=0)
print(zs)

## 2. Strategy B — Few-shot prompting
Same model, but prepend a few labeled **in-context examples** to the prompt.

In [ ]:
shots = train[:2] + train[-2:]  # 2 positive + 2 negative
preamble = 'Classify each product review as positive or negative.\n'
for t, y in shots:
    preamble += f'Review: "{t}"\nAnswer: {LABELS[y]}\n'

def few_shot(text):
    prompt = preamble + f'Review: "{text}"\nAnswer:'
    return parse_sentiment(flan_generate(prompt))

t0 = time.time()
pred_fs = [few_shot(t) for t in test_texts]
fs = dict(strategy='few-shot',
          accuracy=round(accuracy_score(test_labels, pred_fs), 3),
          macro_f1=round(f1_score(test_labels, pred_fs, average='macro'), 3),
          sec=round(time.time()-t0, 1), trained_params=0)
print(fs)

## 3. Strategy C — LoRA fine-tuning
Fine-tune a small encoder (`distilbert-base-uncased`) with a **LoRA adapter** — only a tiny fraction of parameters are trained.

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from peft import LoraConfig, get_peft_model, TaskType
import torch

ckpt = 'distilbert-base-uncased'
tok = AutoTokenizer.from_pretrained(ckpt)

class DS(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.enc = tok([t for t, _ in rows], truncation=True, padding='max_length', max_length=64)
        self.y = [y for _, y in rows]
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item['labels'] = torch.tensor(self.y[i]); return item

base = AutoModelForSequenceClassification.from_pretrained(ckpt, num_labels=2)
lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.05,
                  target_modules=['q_lin', 'v_lin'])
model = get_peft_model(base, lora)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
model.print_trainable_parameters()

args = TrainingArguments(output_dir='/tmp/lora_out', num_train_epochs=8,
                         per_device_train_batch_size=8, learning_rate=2e-3,
                         logging_steps=50, report_to='none', seed=SEED)
trainer = Trainer(model=model, args=args, train_dataset=DS(train))
t0 = time.time(); trainer.train(); train_sec = round(time.time()-t0, 1)

import numpy as np
model.eval()
enc = tok(test_texts, truncation=True, padding=True, max_length=64, return_tensors='pt').to(model.device)
with torch.no_grad():
    pred_lora = model(**enc).logits.argmax(-1).cpu().tolist()
lora_res = dict(strategy='LoRA',
                accuracy=round(accuracy_score(test_labels, pred_lora), 3),
                macro_f1=round(f1_score(test_labels, pred_lora, average='macro'), 3),
                sec=train_sec, trained_params=trainable)
print(lora_res)

## 4. Compare
This table — built from the runs above — drives your analysis. `sec` is wall-clock cost (prompting = inference time; LoRA = training time); `trained_params` shows how little LoRA updates.

In [ ]:
results = pd.DataFrame([zs, fs, lora_res])[['strategy','accuracy','macro_f1','sec','trained_params']]
results

## 5. Analysis (≈500 words)
- **Winner.** Which strategy best fits *your* project, and why? Consider the accuracy/effort/cost tradeoff, not just the top number.
- **When prompting wins.** No training, instant to change — but per-call cost and prompt sensitivity. When is that the right call?
- **When fine-tuning wins.** Upfront cost and data needs, but cheap, consistent inference. Connect to HOLLM Ch. 11–12.
- **Data.** How would more labeled data change your recommendation (Tunstall Ch. 9)?